# T7 It5: blind check on NUDT fresh 606 (scored once, after the freeze)

300 NUDT photos drawn with seed 606 from the 2400-photo pool, with every photo of the seen NUDT sets excluded (checked below). Fine matching was
dumped before any T7 rule existed (`Rich_Dump_Run_executed_nudt_fresh606`), the signal table built with the It1 notebook (`ONLY=nudt606`), and the
rules were frozen in It4 (`results/frozen_rules_t7.json`, file times printed below). This notebook applies them once. Baselines: the same
agreement-free replays as in It2 / It4.

In [1]:
import os, sys, json
os.environ.setdefault("OMP_NUM_THREADS", str(len(os.sched_getaffinity(0))))
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import joblib, numpy as np, pandas as pd
from datetime import datetime
from train.checks.candidate_judge import add_context, add_features, baseline, decide_frozen, judge_new, paired_gain, summary
RES = ROOT / "tests/Test_Iteration_7/results"
for f in ["frozen_rules_t7.json", "judge_t7.joblib", "signals/nudt606.csv.gz"]:
    print(f, datetime.fromtimestamp((RES / f).stat().st_mtime).isoformat(timespec="seconds"))
frozen = json.load(open(RES / "frozen_rules_t7.json")); model = joblib.load(RES / "judge_t7.joblib")
blind = add_context(add_features(pd.read_csv(RES / "signals/nudt606.csv.gz"))).reset_index(drop=True)
# Photo ids index the loaded pool: 404 / 505 / 606 share the 2400-photo pool (ids comparable); 101 / 202 index the 500-photo pool, which the 606 draw
# excluded as a whole (FRESH_EXCLUDE_POOL=500), so their ids are not comparable and need no check.
seen = {s: set(pd.read_csv(RES / f"signals/{s}.csv.gz", usecols=["photo"]).photo) for s in ["nudt404", "nudt505"]}
print("blind photos:", blind.photo.nunique(), "| shared with fresh 404 / 505 (same pool):", {s: len(v & set(blind.photo)) for s, v in seen.items()})

frozen_rules_t7.json 2026-11-13T16:45:24
judge_t7.joblib 2026-11-13T16:45:24
signals/nudt606.csv.gz 2026-11-13T13:39:10


blind photos: 300 | shared with fresh 404 / 505 (same pool): {'nudt404': 0, 'nudt505': 0}


## The frozen rule, applied once

In [2]:
p = judge_new(model, blind)
bases = {"pc": baseline(blind, 8, 11, min_choice=3), "jetson": baseline(blind, 8, 8, jetson=True, m=5)}
rows, per_photo = [], []
for variant, cfg in frozen["variants"].items():
    base = bases["jetson" if cfg.get("jetson") else "pc"]
    r = decide_frozen(blind, p, cfg["tau"], variant)
    s, b = summary(r).iloc[0], summary(base).iloc[0]
    rows.append(dict(variant=variant, cut=cfg["tau"], score=s.score, wrong=s.wrong, answered=s.answered, queries=s.queries,
                     baseline=f"{b.score:.2f} / {b.wrong}w / q{b.queries:.2f}", gain=paired_gain(base, r, ["nudt606"])["nudt606"]))
    per_photo.append(r.assign(variant=variant))
res = pd.DataFrame(rows)
res["modelled Jetson s / photo"] = (res.queries * 2.2).round(1)
display(res)
res.to_csv(RES / "it5_blind606.csv", index=False)
pd.concat(per_photo).to_csv(RES / "it5_blind606_photos.csv", index=False)

,variant,cut,score,wrong,answered,queries,baseline,gain,modelled Jetson s / photo
0,pc,0.9,55.80,2.0,0.963,7.540,55.08 / 2.0w / q7.54,"+0.72 [-0.56, +2.12]",16.6
1,pc_early,0.9,55.48,2.0,0.963,3.140,55.08 / 2.0w / q7.54,"+0.40 [-0.92, +1.84]",6.9
2,jetson,0.9,54.82,1.0,0.930,6.303,54.42 / 2.0w / q6.29,"+0.40 [-0.60, +1.60]",13.9
3,jetson_early,0.9,54.50,1.0,0.930,1.903,54.42 / 2.0w / q6.29,"+0.08 [-0.96, +1.32]",4.2


## Photos that change against the baseline

In [3]:
for variant in ["pc", "jetson"]:
    base = bases[variant].set_index("photo"); r = per_photo[list(frozen["variants"]).index(variant)].set_index("photo")
    j = base.join(r, rsuffix="_judge")
    ch = j[j.pts != j.pts_judge][["stage", "pts", "inliers", "stage_judge", "pts_judge", "inliers_judge", "p_judge", "err_sigma_judge"]]
    print(variant, "| gained", int((ch.pts_judge > ch.pts).sum()), "| lost", int((ch.pts_judge < ch.pts).sum()),
          "| wrong removed", int(((ch.pts < 0) & (ch.pts_judge >= 0)).sum()), "| wrong added", int(((ch.pts >= 0) & (ch.pts_judge < 0)).sum()))
    display(ch.round(2))

pc | gained 5 | lost 1 | wrong removed 1 | wrong added 1


,stage,pts,inliers,stage_judge,pts_judge,inliers_judge,p_judge,err_sigma_judge
photo,,,,,,,,
6,skip,0,7.0,std6,10,4.0,0.98,0.42
169,skip,0,10.0,wide,10,10.0,1.00,0.56
254,std6,-20,8.0,skip,0,9.0,0.12,964.26
1233,skip,0,10.0,wide,6,10.0,0.99,1.01
1264,skip,0,10.0,wide,10,10.0,0.99,0.85
2071,skip,0,3.0,wide,-20,3.0,0.95,88.34


jetson | gained 3 | lost 2 | wrong removed 1 | wrong added 0


,stage,pts,inliers,stage_judge,pts_judge,inliers_judge,p_judge,err_sigma_judge
photo,,,,,,,,
6,skip,0,7.0,std6,10,4.0,0.98,0.42
189,wide,10,8.0,skip,0,8.0,0.93,0.46
254,std6,-20,8.0,skip,0,9.0,0.12,964.26
368,std6,10,8.0,skip,0,8.0,0.94,0.87
1233,skip,0,5.0,wide,10,5.0,0.95,0.45


## Reading

- **Wrong answers:** PC 2 (baseline 2): the judge removes photo 254 (8 inliers, P 0.12) but adds photo 2071 (3 inliers, DINO z 2.64, P 0.95, wide
  queries); photo 182 (8 inliers, P 0.96) is wrong under both. Jetson 1 (baseline 2): 254 removed, 182 left.
- **Score:** PC 55.80 vs 55.08 (+0.72 [-0.56, +2.12]), Jetson 54.82 vs 54.42 (+0.40 [-0.60, +1.60]). Positive, but the intervals include 0, and the
  wrong count is not 0: by the brief's rule this is **not** a proven gain on the blind set.
- **Early exit holds:** Jetson early exit 54.50 vs 54.42 at 1.9 instead of 6.3 LoFTR queries per photo (modelled 4.2 s vs 13.9 s).
- Follow-up in It6 (seen sets only): a minimum of 4 inliers for an answer would make the 2071 kind of answer impossible at ~0.2 points cost.